# 08. MDP와 불확실성 하의 의사결정

Probabilistic Robotics 후반부는 perception만이 아니라 planning/control도 확률적으로 다룬다.
MDP는 상태 전이의 불확실성을 포함한 의사결정 모델이다.

$$V(s)=\max_a \left[ R(s,a)+\gamma\sum_{s'}p(s'\mid s,a)V(s') \right]$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Grid World Value Iteration

명령한 방향으로 80% 이동하고, 20%는 좌우로 미끄러지는 grid world를 푼다.

In [ ]:
H,W=7,9
goal=(5,7); obstacle={(2,3),(3,3),(4,3),(1,6),(2,6)}
actions=[(-1,0),(1,0),(0,-1),(0,1)]
chars=['↑','↓','←','→']
gamma=0.95

def step(s,a):
    if s==goal: return s
    ns=(s[0]+a[0],s[1]+a[1])
    if ns[0]<0 or ns[0]>=H or ns[1]<0 or ns[1]>=W or ns in obstacle:
        return s
    return ns

def transitions(s,ai):
    if s==goal: return [(1.0,s)]
    left=[2,3,1,0][ai]; right=[3,2,0,1][ai]
    probs=[(0.8,ai),(0.1,left),(0.1,right)]
    out={}
    for p,j in probs:
        ns=step(s,actions[j]); out[ns]=out.get(ns,0)+p
    return [(p,ns) for ns,p in out.items()]

V=np.zeros((H,W))
for it in range(200):
    Vnew=V.copy()
    for r in range(H):
        for c in range(W):
            s=(r,c)
            if s in obstacle: continue
            if s==goal:
                Vnew[s]=10; continue
            qs=[]
            for ai in range(4):
                q=-0.1 + gamma*sum(p*V[ns] for p,ns in transitions(s,ai))
                qs.append(q)
            Vnew[s]=max(qs)
    if np.max(np.abs(Vnew-V))<1e-5: break
    V=Vnew
policy=np.full((H,W),' ',dtype='<U1')
for r in range(H):
    for c in range(W):
        s=(r,c)
        if s in obstacle: policy[s]='■'
        elif s==goal: policy[s]='G'
        else:
            qs=[-0.1+gamma*sum(p*V[ns] for p,ns in transitions(s,ai)) for ai in range(4)]
            policy[s]=chars[int(np.argmax(qs))]

fig, axes=plt.subplots(1,2,figsize=(12,5))
im=axes[0].imshow(V,cmap='viridis')
for (r,c) in obstacle: axes[0].add_patch(plt.Rectangle((c-0.5,r-0.5),1,1,color='black'))
axes[0].scatter(goal[1],goal[0],marker='*',s=180,color='#E85D24')
axes[0].set_title('Value function'); plt.colorbar(im,ax=axes[0],fraction=0.046)
axes[1].imshow(np.zeros((H,W)),cmap='gray',vmin=0,vmax=1)
for r in range(H):
    for c in range(W):
        axes[1].text(c,r,policy[r,c],ha='center',va='center',fontsize=18,color='white' if policy[r,c]=='■' else 'black')
axes[1].set_title('Optimal policy under stochastic transitions')
for ax in axes:
    ax.set_xticks(range(W)); ax.set_yticks(range(H)); ax.grid(color='white',lw=1.2); ax.set_xlim(-0.5,W-0.5); ax.set_ylim(H-0.5,-0.5)
plt.tight_layout(); plt.savefig('assets/08_mdp_value_iteration.png',dpi=150,bbox_inches='tight'); plt.show()
print('iterations:', it+1)

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| MDP | 완전 관측 상태에서 확률적 planning | Ch.14 Markov Decision Processes |
| Value iteration | Bellman backup 반복 | 최적 policy 계산 |
| Transition uncertainty | 명령 실패/미끄러짐 모델 | 로봇 행동의 불확실성 |
| POMDP | belief를 상태로 쓰는 planning | Ch.15-16 POMDP |